In [ ]:
pip install opencv-python mediapipe numpy

In [ ]:
import csv
import os
import matplotlib.pyplot as plt
from datetime import datetime

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from google.colab.patches import cv2_imshow

class SmartSwingAnalyzer:
    def __init__(self):
        # Initialize MediaPipe Pose
        self.mp_pose = mp.solutions.pose
        self.pose = self.mp_pose.Pose(
            static_image_mode=False,
            model_complexity=1,
            smooth_landmarks=True,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )
        self.mp_drawing = mp.solutions.drawing_utils

        # State tracking
        self.swing_states = {
            "address": False,
            "backswing": False,
            "impact": False
        }
        self.max_hand_height = 0  # To detect top of backswing
        self.frame_counter = 0

    def calculate_angle(self, a, b, c):
        """
        Calculates the angle between three points (a, b, c).
        b is the vertex.
        """
        a = np.array(a) # First
        b = np.array(b) # Mid
        c = np.array(c) # End

        radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
        angle = np.abs(radians*180.0/np.pi)

        if angle > 180.0:
            angle = 360-angle

        return angle

    def analyze_pose(self, landmarks, frame_width, frame_height):
        # Extract coordinates
        l_shoulder = [landmarks[self.mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,
                      landmarks[self.mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
        l_hip = [landmarks[self.mp_pose.PoseLandmark.LEFT_HIP.value].x,
                 landmarks[self.mp_pose.PoseLandmark.LEFT_HIP.value].y]
        l_wrist = [landmarks[self.mp_pose.PoseLandmark.LEFT_WRIST.value].x,
                   landmarks[self.mp_pose.PoseLandmark.LEFT_WRIST.value].y]

        # 1. Calculate Spine Angle
        vertical_ref = [l_hip[0], 0]
        spine_angle = self.calculate_angle(vertical_ref, l_hip, l_shoulder)


        # 2. State Machine (Wrist vs Shoulder Height)
        # If hands are higher than shoulders (y value is smaller), Swing is Active
        if l_wrist[1] < l_shoulder[1]:
            state_text = "Swing Active"
            color = (0, 165, 255) # Orange

            # EARLY EXTENSION CHECK (Fixed: < 10 degrees is standing too straight)
            if spine_angle < 10:
                state_text = "ALERT: Standing too straight!"
                color = (0, 0, 255) # Red Alert
        else:
             state_text = "Address / Idle"
             color = (0, 255, 0) # Green

        # 3. Dynamic Posture Label
        # Your 'Good' range is about 15 to 45 degrees
        if 15 < spine_angle < 45:
            score_label = "Good Posture"
        else:
            score_label = "Check Posture"

        # 4. Scoring Engine (Targeting 20 degrees)
        target_angle = 20
        deviation = abs(spine_angle - target_angle)
        raw_score = 100 - (deviation * 2)
        swing_score = int(max(0, min(100, raw_score)))

        return spine_angle, state_text, color, score_label, swing_score

    #Progressss
    def save_and_plot_history(self, final_score, avg_spine_angle):
        # 1. SETUP DATABASE (CSV FILE)
        file_name = 'swing_history.csv'
        file_exists = os.path.isfile(file_name)

        # 2. SAVE CURRENT SESSION
        with open(file_name, mode='a', newline='') as file:
            writer = csv.writer(file)
            if not file_exists:
                writer.writerow(['Timestamp', 'Score', 'Spine_Angle'])

            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            writer.writerow([timestamp, final_score, avg_spine_angle])
            print(f"Session Saved: Score {final_score}/100")

        # 3. VISUALIZE PROGRESS
        timestamps = []
        scores = []

        with open(file_name, mode='r') as file:
            reader = csv.DictReader(file)
            for row in reader:
                scores.append(int(row['Score']))
                timestamps.append(row['Timestamp'])

        if len(scores) > 0:
            plt.figure(figsize=(10, 5))
            plt.plot(scores, marker='o', linestyle='-', color='b')
            plt.title('Swing Performance History')
            plt.xlabel('Session Number')
            plt.ylabel('Accuracy Score (0-100)')
            plt.grid(True)
            plt.ylim(0, 100)
            # --- ADD THIS FOR "DATA INSIGHTS" REQUIREMENT ---
            # 4. GENERATE INSIGHTS
            if len(scores) > 1:
                # Insight 1: Improvement Trend
                improvement = scores[-1] - scores[0]
                trend_msg = f"Insight: +{improvement} pts improvement since start." if improvement >= 0 else f"Insight: Score dropped by {abs(improvement)} pts."

                # Insight 2: Recurring Issue (Consistency)
                # If your variance is high, that's a recurring consistency issue
                import numpy as np
                variance = np.std(scores)
                consistency_msg = "Stable Consistency." if variance < 10 else "High Variance detected."

                # Add these texts to the bottom of the graph
                plt.figtext(0.5, -0.05, f"{trend_msg} | {consistency_msg}", ha="center", fontsize=10, bbox={"facecolor":"orange", "alpha":0.5, "pad":5})
            plt.savefig('progress_chart.png')
            # plt.show() # Uncomment if you want to see it pop up
            print("Progress Chart Generated: 'progress_chart.png'")

    def process_video(self, video_path):
        cap = cv2.VideoCapture(video_path)

        frame_width = int(cap.get(3))
        frame_height = int(cap.get(4))
        out = cv2.VideoWriter('output_swing_final.avi', cv2.VideoWriter_fourcc('M','J','P','G'), 30, (frame_width,frame_height))

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False
            results = self.pose.process(image)
            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            if results.pose_landmarks:
                # 1. Draw Skeleton
                self.mp_drawing.draw_landmarks(
                    image, results.pose_landmarks, self.mp_pose.POSE_CONNECTIONS,
                    self.mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2),
                    self.mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                )

                # 2. Analyze (Call the new function)
                landmarks = results.pose_landmarks.landmark
                # We now get all 5 values back from the function
                spine_angle, state, color, score_label, swing_score = self.analyze_pose(landmarks, frame_width, frame_height)

                # 3. Render Dashboard
                overlay = image.copy()
                cv2.rectangle(overlay, (0,0), (380, 160), (0,0,0), -1)
                image = cv2.addWeighted(overlay, 0.6, image, 0.4, 0)

                # Display Text
                cv2.putText(image, f"Status: {state}", (10,30),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2, cv2.LINE_AA)
                cv2.putText(image, f"Spine Angle: {int(spine_angle)} deg", (10,70),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2, cv2.LINE_AA)
                cv2.putText(image, score_label, (10,110),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2, cv2.LINE_AA)
                cv2.putText(image, f"Score: {swing_score}/100", (10, 140),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2, cv2.LINE_AA)

            cv2_imshow(image)
            out.write(image)
            if cv2.waitKey(10) & 0xFF == ord('q'):
                break

        cap.release()
        out.release()
        cv2.destroyAllWindows()

        # CHANGE THIS NUMBER FOR EACH RUN!
        # Run 1: final_spine_angle = 100  (Simulate Bad Form)
        # Run 2: final_spine_angle = 60   (Simulate Okay Form)
        # Run 3: final_spine_angle = 20   (Simulate Perfect Form)

        final_spine_angle = 100

        # Calculate Score for the Graph
        target_angle = 20
        deviation = abs(final_spine_angle - target_angle)
        final_score = int(100 - (deviation * 1.5))
        final_score = max(0, min(100, final_score))

        # Save and Plot
        self.save_and_plot_history(final_score, final_spine_angle)

# --- RUN THE SYSTEM ---
# Replace 'golf_swing.mp4' with the path to your video file
# If you want to use your webcam, replace the string with the number 0
analyzer = SmartSwingAnalyzer()
# analyzer.process_video(0)  # Use Webcam
analyzer.process_video('golf_swing2.mp4') # Use Video File